In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error


In [3]:
test = pd.read_csv('test.csv')
train = pd.read_csv('train.csv')
train.head()
train.info()
print(test.columns.tolist())
print(test.shape)

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       180 non-null    int64  
 1   Gender                   180 non-null    str    
 2   Age                      180 non-null    int64  
 3   Occupation               180 non-null    str    
 4   Physical_Activity_Level  180 non-null    int64  
 5   Stress_Level             180 non-null    int64  
 6   BMI_Category             180 non-null    str    
 7   Heart_Rate               180 non-null    int64  
 8   Daily_Steps              180 non-null    int64  
 9   Sleep_Disorder           80 non-null     str    
 10  Temp                     180 non-null    float64
 11  Mood                     180 non-null    float64
 12  History                  180 non-null    float64
 13  Quality_of_Sleep         180 non-null    float64
 14  Sleep_Duration           180 non-null

In [4]:
#read dataset
train['Sleep_Disorder'] = train["Sleep_Disorder"].fillna("None")
test['Sleep_Disorder'] = test ['Sleep_Disorder'].fillna("None")
print(train['Sleep_Disorder'].value_counts())

Sleep_Disorder
None           100
Sleep Apnea     47
Insomnia        33
Name: count, dtype: int64


In [5]:
# Modify parameters 
target = 'Sleep_Duration'
id_col = 'ID'
x = train.drop(columns = [id_col,target])
y = train[target]
x_test = test.drop(columns = [id_col])
num_col = x.select_dtypes(include ='number').columns.tolist()
cat_col = x.select_dtypes(exclude = 'number').columns.tolist()
print(x.shape,y.shape,x_test.shape)
print("numbers:", num_col)
print("categoric_Label:", cat_col)

(180, 13) (180,) (194, 13)
numbers: ['Age', 'Physical_Activity_Level', 'Stress_Level', 'Heart_Rate', 'Daily_Steps', 'Temp', 'Mood', 'History', 'Quality_of_Sleep']
categoric_Label: ['Gender', 'Occupation', 'BMI_Category', 'Sleep_Disorder']


In [6]:
# generalize all data and pipeline all dataset which can be used directly in predictiona
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_col),
    ("cat", OneHotEncoder(handle_unknown = "ignore"), cat_col)
    ])
linearRegression_model = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression())
    ])

In [7]:
# linear regression prediction and measure RMSE which is used comparing with deviation
x_tr_L, x_val_L, y_tr_L, y_val_L = train_test_split(x, y, test_size=0.2, random_state=42)
linearRegression_model.fit(x_tr_L, y_tr_L)
pred_val = linearRegression_model.predict(x_val_L)
rmse = np.sqrt(mean_squared_error(y_val_L, pred_val))
scores = cross_val_score(linearRegression_model, x, y, cv=5, scoring="neg_root_mean_squared_error")
print("RMSE:", rmse)
print(y.std()) # compare deviation with rmse,which shows the much lower value, due to the reduction of error

RMSE: 0.2761320227747438
0.7975776096727343


In [10]:
# Shirkage method: Ridge: error + alpha*sum(coeff**2) and Lasso: error + alpha * sum(abs(coeff))
Ridge_model = Pipeline([
    ("prep", preprocess),
    ("Rig", RidgeCV(alphas=[0.001,0.01,0.1,1]))
])
Lossa_model = Pipeline([
    ("prep", preprocess),
    ("Lass", LassoCV(cv = 5, random_state = 0))
])

x_tr, x_val, y_tr, y_val = train_test_split(x, y, test_size=0.2, random_state=42)
#print(len(x_tr), len(x_val), len(y_tr), len(y_val))
for name, model in [("Ridge", Ridge_model),("Lossa", Lossa_model)]:
    model.fit(x_tr, y_tr)
    pred_val = model.predict(x_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred_val))
    print(f"{name}: RMSE = {rmse}")
Ridge_model.fit(x,y)
pred_val_y = Ridge_model.predict(x_test)
df = {"ID":test['ID'],"Sleep_Duration":pred_val_y}
df = pd.DataFrame(df)
df.to_csv('sample_submission_shirkage_methods.csv', index = False)
print(train.select_dtypes("number").corr()["Sleep_Duration"].sort_values())
print(y.describe())
print(y.std())

Ridge: RMSE = 0.2607058300578689
Lossa: RMSE = 0.28525178844135674
Stress_Level              -0.816001
Heart_Rate                -0.477468
ID                        -0.125668
Daily_Steps               -0.102529
Physical_Activity_Level    0.177873
Age                        0.350265
Temp                       0.389189
History                    0.555558
Quality_of_Sleep           0.571502
Mood                       0.634548
Sleep_Duration             1.000000
Name: Sleep_Duration, dtype: float64
count    180.000000
mean       7.176111
std        0.797578
min        5.800000
25%        6.500000
50%        7.200000
75%        7.800000
max        8.500000
Name: Sleep_Duration, dtype: float64
0.7975776096727343


Ridge shrikage is used to predict the sleep duration due to the rmse = 0.26 which is the root of mean square error. Mean prediction value is 7.17h, and the range is from 5.8~8.5h. 
According to the correlation results, Stress level and Mood are highly relevant with duration of sleeping. 

In [9]:
# Foward and backward stepwise method
forward_model = Pipeline([
    ("prep",preprocess),
    ("stepwise", SequentialFeatureSelector(estimator = RidgeCV(),
                                           n_features_to_select='auto', 
                                            direction = 'forward',
                                            scoring = "neg_root_mean_squared_error",
                                            cv = 5)),
    ("reg", RidgeCV())])

backward_model = Pipeline([
    ("prep",preprocess),
    ("stepwise", SequentialFeatureSelector(estimator = RidgeCV(),
                                            n_features_to_select='auto', 
                                            direction = 'backward',
                                            scoring = "neg_mean_squared_error",
                                            cv = 5)),
    ("reg", RidgeCV())])

x_tr_s, x_val_s, y_tr_s, y_val_s = train_test_split(x, y, test_size=0.2, random_state=42)
for name, model in [('forward', forward_model),('backward',backward_model)]:
    model.fit(x_tr_s, y_tr_s)
    pred_val_s = model.predict(x_val_s)
    mse_s = mean_squared_error(y_val_s, pred_val_s)
    print(f"{name}: RMSE = {mse_s}")
forward_model.fit(x,y)
pred_val_y_s = forward_model.predict(x_test)
df = {"ID":test['ID'],"Sleep_Duration":pred_val_y_s}
df = pd.DataFrame(df)
df.to_csv('sample_submission_stepwise_methods.csv', index = False)

forward: RMSE = 0.06333327534664883
backward: RMSE = 0.0736588813381091
